# Product Price Estimation via Fine-Tuned Language Models
This notebook constructs a price prediction pipeline using Amazon product metadata. It covers dataset assembly, feature engineering, classical ML baselines, and GPT-4o-mini fine-tuning for end-to-end price estimation.

In [ ]:
# Install required packages not available in the default environment using uv
!uv pip install -q gensim
!uv pip install -q --upgrade datasets==3.6.0
!uv pip install -q --upgrade huggingface-hub==0.34.0


In [ ]:
# Standard library
import os
import math
import random
import json
import pickle
import re
from datetime import datetime
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from tqdm import tqdm
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.svm import LinearSVR
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict, load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# Load environment variables and authenticate with Hugging Face
load_dotenv(override=True)
openai_key = os.environ.get("OPENAI_API_KEY")
hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    print("Authenticating with Hugging Face...")
    login(hf_token, add_to_git_credential=True)
else:
    print("Warning: HF_TOKEN not set. Dataset access may be restricted.")

In [ ]:
# ANSI terminal colors for per-prediction output quality coloring
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
RESET  = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}

In [ ]:
# Model and tokenization configuration
BASE_MODEL   = "meta-llama/Meta-Llama-3.1-8B"
MIN_CHARS    = 300
MIN_TOKENS   = 150
MAX_TOKENS   = 160
CEILING_CHARS = MAX_TOKENS * 7  # Rough character cap before tokenization


class Item:
    """
    Represents a single Amazon product entry.

    Parses raw product metadata into a structured prompt suitable for
    both fine-tuning and inference. Items that lack sufficient text
    or fall outside acceptable token bounds are marked for exclusion.
    """

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    PREFIX    = "Price is $"
    QUESTION  = "How much does this cost to the nearest dollar?"

    # Detail field fragments to strip before building prompts
    REMOVALS = [
        '"Batteries Included?": "No"', '"Batteries Included?": "Yes"',
        '"Batteries Required?": "No"', '"Batteries Required?": "Yes"',
        "By Manufacturer", "Item", "Date First", "Package",
        ":", "Number of", "Best Sellers", "Number", "Product "
    ]

    def __init__(self, data, price):
        self.title       = data["title"]
        self.price       = price
        self.category    = data.get("category", "Unknown")
        self.token_count = 0
        self.details     = None
        self.prompt      = None
        self.include     = False
        self.parse(data)

    def scrub_details(self):
        """Remove noisy boilerplate from the raw details string."""
        details = self.details
        for token in self.REMOVALS:
            details = details.replace(token, "")
        return details

    def scrub(self, text):
        """Normalize whitespace, punctuation, and long numeric tokens."""
        text = re.sub(r'[:\[\]"{}\u3010\u3011\s]+', ' ', text).strip()
        text = text.replace(" ,", ",").replace(",,,", ",").replace(",,", ",")
        # Drop long tokens that are mostly digits (e.g. part numbers)
        words = text.split(" ")
        select = [w for w in words if len(w) < 7 or not any(c.isdigit() for c in w)]
        return " ".join(select)

    def parse(self, data):
        """Assemble and validate the product text, then build the prompt."""
        contents = '\n'.join(data.get("description", []))
        if contents:
            contents += '\n'

        features = '\n'.join(data.get("features", []))
        if features:
            contents += features + '\n'

        self.details = data.get("details")
        if self.details:
            contents += self.scrub_details() + '\n'

        if len(contents) > MIN_CHARS:
            contents = contents[:CEILING_CHARS]
            text   = f"{self.scrub(self.title)}\n{self.scrub(contents)}"
            tokens = self.tokenizer.encode(text, add_special_tokens=False)

            if len(tokens) > MIN_TOKENS:
                tokens = tokens[:MAX_TOKENS]
                text   = self.tokenizer.decode(tokens)
                self.make_prompt(text)
                self.include = True

    def make_prompt(self, text):
        """Construct the full training prompt including the price answer."""
        self.prompt  = f"{self.QUESTION}\n\n{text}\n\n"
        self.prompt += f"{self.PREFIX}{round(self.price)}.00"
        self.token_count = len(self.tokenizer.encode(self.prompt, add_special_tokens=False))

    def test_prompt(self):
        """Return the prompt without the answer, for inference use."""
        return self.prompt.split(self.PREFIX)[0] + self.PREFIX

    def __repr__(self):
        return f"<{self.title} = ${self.price}>"

In [ ]:
# Price filter bounds and chunk size for parallel loading
MIN_PRICE  = 0.5
MAX_PRICE  = 999.49
CHUNK_SIZE = 1000


class ItemLoader:
    """
    Streams and filters Amazon product records from a HuggingFace dataset.

    Items outside the price range or with insufficient text are discarded.
    Parallel processing via ProcessPoolExecutor speeds up large categories.
    """

    def __init__(self, name):
        self.name    = name
        self.dataset = None

    def from_datapoint(self, datapoint):
        """Attempt to construct an Item from a single record; return None if invalid."""
        try:
            price_str = datapoint.get("price")
            if price_str:
                price = float(price_str)
                if MIN_PRICE <= price <= MAX_PRICE:
                    item = Item(datapoint, price)
                    if item.include:
                        return item
        except ValueError:
            pass
        return None

    def from_chunk(self, chunk):
        """Process a dataset slice and return valid Items."""
        return [item for dp in chunk if (item := self.from_datapoint(dp)) is not None]

    def chunk_generator(self):
        """Yield successive fixed-size slices of the dataset."""
        size = len(self.dataset)
        for start in range(0, size, CHUNK_SIZE):
            yield self.dataset.select(range(start, min(start + CHUNK_SIZE, size)))

    def load_in_parallel(self, workers):
        """Distribute chunk processing across worker processes."""
        results    = []
        chunk_count = (len(self.dataset) // CHUNK_SIZE) + 1

        with ProcessPoolExecutor(max_workers=workers) as pool:
            for batch in tqdm(
                pool.map(self.from_chunk, self.chunk_generator()),
                total=chunk_count,
                desc=self.name
            ):
                results.extend(batch)

        for result in results:
            result.category = self.name

        return results

    def load(self, workers=8):
        """Fetch the dataset from HuggingFace and process it in parallel."""
        self.dataset = load_dataset(
            "McAuley-Lab/Amazon-Reviews-2023",
            f"raw_meta_{self.name}",
            split="full",
            trust_remote_code=True
        )
        start   = datetime.now()
        print(f"Loaded dataset: {self.dataset}")
        results  = self.load_in_parallel(workers)
        duration = (datetime.now() - start).total_seconds() / 60
        print(f"Finished '{self.name}': {len(results):,} items in {duration:.1f} min")
        return results

In [ ]:
class Tester:
    """
    Evaluates a price predictor against a labeled test set.

    Metrics reported:
      - Mean absolute error (MAE)
      - Root mean squared log error (RMSLE)
      - Hit rate: percentage of predictions within $40 or 20% of true price

    Results are also visualized as a scatter plot of predicted vs. actual prices.
    """

    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data      = data
        self.title     = title or predictor.__name__.replace("_", " ").title()
        self.size      = size
        self.guesses   = []
        self.truths    = []
        self.errors    = []
        self.sles      = []
        self.colors    = []

    def color_for(self, error, truth):
        """Assign a quality label based on absolute and relative error."""
        if error < 40 or error / truth < 0.2:
            return "green"
        if error < 80 or error / truth < 0.4:
            return "orange"
        return "red"

    def run_datapoint(self, index):
        """Run prediction on one item and record all relevant metrics."""
        datapoint = self.data[index]
        guess     = self.predictor(datapoint)
        truth     = datapoint.price
        error     = abs(guess - truth)
        log_error = math.log(truth + 1) - math.log(guess + 1)
        sle       = log_error ** 2
        color     = self.color_for(error, truth)
        name      = datapoint.title[:40] + "..." if len(datapoint.title) > 40 else datapoint.title

        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)

        print(
            f"{COLOR_MAP[color]}{index + 1}: "
            f"Guess: ${guess:,.2f}  Truth: ${truth:,.2f}  "
            f"Error: ${error:,.2f}  SLE: {sle:,.2f}  Item: {name}{RESET}"
        )

    def chart(self, title):
        """Plot predicted vs. actual prices with a reference diagonal."""
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color="deepskyblue", lw=2, alpha=0.6, label="Perfect prediction")
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel("Actual Price ($)")
        plt.ylabel("Predicted Price ($)")
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.legend()
        plt.tight_layout()
        plt.show()

    def report(self):
        """Compute aggregate metrics and render the chart."""
        mae   = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits  = sum(1 for c in self.colors if c == "green")
        title = f"{self.title}  MAE=${mae:,.2f}  RMSLE={rmsle:,.2f}  Hits={hits / self.size * 100:.1f}%"
        self.chart(title)

    def run(self):
        """Evaluate the predictor over all test items and report results."""
        for index in range(self.size):
            self.run_datapoint(index)
        self.report()

    @classmethod
    def test(cls, function, data):
        """Convenience method: instantiate and run in one call."""
        cls(function, data).run()


def get_price(s):
    """Extract the first floating-point number from a price string."""
    s = s.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", s)
    return float(match.group()) if match else 0.0

## Dataset Assembly

### Ingesting Product Catalogs
Load and filter items from selected Amazon product categories. Uncomment additional categories to expand the training corpus.

In [ ]:
catalog_labels = [
    "All_Beauty",
    # "Automotive",
    # "Electronics",
    # "Office_Products",
    # "Tools_and_Home_Improvement",
    # "Cell_Phones_and_Accessories",
    # "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
    "Software",
    "Handmade_Products"
]

curated_pool = []
for label in catalog_labels:
    print(f"Ingesting: {label}")
    curated_pool.extend(ItemLoader(label).load())

print(f"\nTotal items in pool: {len(curated_pool):,}")

In [ ]:
# Summarize price and token distributions across the full pool
price_series    = [item.price for item in curated_pool]
token_series    = [item.token_count for item in curated_pool]
category_tally  = Counter(item.category for item in curated_pool)
summary_frame   = pd.DataFrame({"price": price_series, "tokens": token_series})

display(summary_frame.describe())
display(
    pd.DataFrame.from_dict(category_tally, orient="index", columns=["count"])
    .sort_values("count", ascending=False)
)

In [ ]:
# Bucket items by rounded integer price for balanced sampling
price_slots = defaultdict(list)
for item in curated_pool:
    key = round(item.price)
    if 1 <= key <= 999:
        price_slots[key].append(item)

print(f"Distinct price slots: {len(price_slots)}")

In [ ]:
# Build a balanced training bundle:
# - High-price items (>= $240) are fully retained (they are naturally sparse)
# - Low-price slots with <= 1200 items are kept in full
# - Overrepresented slots are down-sampled, with Automotive items de-prioritized
random.seed(123)
np.random.seed(123)

SAMPLE_CAP = 1200
HIGH_PRICE_THRESHOLD = 240

balanced_bundle = []
for price in range(1, 1000):
    bucket = price_slots.get(price, [])

    if price >= HIGH_PRICE_THRESHOLD or len(bucket) <= SAMPLE_CAP:
        balanced_bundle.extend(bucket)
    else:
        # Automotive items are over-represented at low prices; reduce their weight
        weights = np.array(
            [1 if item.category == "Automotive" else 5 for item in bucket],
            dtype=float
        )
        weights /= weights.sum()
        indices = np.random.choice(len(bucket), size=SAMPLE_CAP, replace=False, p=weights)
        balanced_bundle.extend(bucket[i] for i in indices)

print(f"Balanced bundle size: {len(balanced_bundle):,}")

In [ ]:
# Verify price and category distributions after balancing
bundle_prices     = [item.price for item in balanced_bundle]
bundle_tokens     = [item.token_count for item in balanced_bundle]
bundle_categories = Counter(item.category for item in balanced_bundle)

display(pd.Series(bundle_prices).describe())
display(
    pd.DataFrame.from_dict(bundle_categories, orient="index", columns=["count"])
    .sort_values("count", ascending=False)
)

In [ ]:
# Visualize price and token distributions across the balanced bundle
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(bundle_prices, bins=range(0, 1000, 10), color="midnightblue", rwidth=0.8)
axes[0].set_xlabel("Price ($)")
axes[0].set_ylabel("Item Count")
axes[0].set_title("Price Distribution")

axes[1].hist(bundle_tokens, bins=range(0, 300, 10), color="forestgreen", rwidth=0.8)
axes[1].set_xlabel("Token Count")
axes[1].set_ylabel("Item Count")
axes[1].set_title("Prompt Length Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Shuffle and split into train / test sets
# Test set is capped at 2000 items; train set at 400,000
random.seed(123)
random.shuffle(balanced_bundle)

test_target  = min(2000, max(1, len(balanced_bundle) // 20))
train_target = min(400_000, len(balanced_bundle) - test_target)
train_items  = balanced_bundle[:train_target]
test_items   = balanced_bundle[train_target:train_target + test_target]

print(f"Training set : {len(train_items):,}")
print(f"Test set     : {len(test_items):,}")

In [ ]:
# Extract prompts and price labels for train and test splits
train_prompts = [item.prompt for item in train_items]
train_prices  = [item.price  for item in train_items]
test_prompts  = [item.test_prompt() for item in test_items]
test_prices   = [item.price  for item in test_items]

In [ ]:
# Package into a HuggingFace DatasetDict for convenient downstream use
train_dataset  = Dataset.from_dict({"text": train_prompts, "price": train_prices})
test_dataset   = Dataset.from_dict({"text": test_prompts,  "price": test_prices})
pricing_dataset = DatasetDict({"train": train_dataset, "test": test_dataset})

### Persisting Datasets to Disk

In [ ]:
# Serialize Item objects via pickle for fast reload in future sessions
storage_dir = Path("data")
storage_dir.mkdir(exist_ok=True)

for name, items in [("balanced_train.pkl", train_items), ("balanced_test.pkl", test_items)]:
    with open(storage_dir / name, "wb") as f:
        pickle.dump(items, f)
    print(f"Saved {name}")

In [ ]:
# Also write Parquet files for use with Arrow-based tooling
pricing_dataset["train"].to_parquet(storage_dir / "balanced_train.parquet")
pricing_dataset["test"].to_parquet(storage_dir / "balanced_test.parquet")
print("Parquet files written.")

## Baseline Models
We establish several reference points before applying fine-tuned models, ranging from random guessing to classical ML regressors.

### Baseline 1: Random Uniform Guess

In [ ]:
# Lower bound: uniformly random price in $1-$999
def stochastic_anchor(item):
    return random.randrange(1, 1000)

random.seed(123)
Tester.test(stochastic_anchor, test_items[:250])

### Baseline 2: Training Set Mean

In [ ]:
# Predict the training mean for every item -- a simple constant predictor
train_price_values = [item.price for item in train_items]
global_mean_price  = sum(train_price_values) / len(train_price_values)

def global_mean_estimator(item):
    return global_mean_price

Tester.test(global_mean_estimator, test_items[:250])

In [ ]:
# Parse the JSON-encoded details field into a dict for feature extraction
def parse_features(raw):
    if not raw:
        return {}
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}

for item in train_items + test_items:
    item.features = parse_features(item.details)

### Structured Feature Engineering
Derive numeric signals from product metadata: item weight, sales rank, text length, and brand tier.

In [ ]:
# Conversion factors for normalizing weight to pounds
WEIGHT_CONVERSIONS = {
    "pounds":     1.0,
    "ounces":     1 / 16,
    "grams":      1 / 453.592,
    "milligrams": 1 / 453592,
    "kilograms":  1 / 0.453592,
}


def infer_weight(item):
    """Return item weight in pounds, or None if unavailable/unparseable."""
    payload = item.features.get("Item Weight")
    if not payload:
        return None

    parts  = payload.split(" ")
    unit   = parts[1].lower() if len(parts) > 1 else ""

    try:
        amount = float(parts[0])
    except (ValueError, IndexError):
        return None

    # Handle edge case: "hundredths pounds"
    if unit == "hundredths" and len(parts) > 2 and parts[2].lower() == "pounds":
        return amount / 100

    return amount * WEIGHT_CONVERSIONS.get(unit, 0) or None

In [ ]:
def infer_rank(item):
    """Return the mean Best Sellers Rank across all sub-categories, or None."""
    payload = item.features.get("Best Sellers Rank")
    if not payload or not isinstance(payload, dict):
        return None
    values = list(payload.values())
    return sum(values) / len(values) if values else None


# Known premium brands -- items from these brands may carry a price premium
TOP_BRANDS = {"nvidia", "hp", "dell", "lenovo", "samsung", "asus", "sony", "canon", "apple", "intel"}


def is_top_brand(item):
    """Return 1 if the item's brand is in the top-brand set, else 0."""
    brand = item.features.get("Brand")
    return 1 if brand and brand.lower() in TOP_BRANDS else 0

In [ ]:
# Compute fallback (mean) values from training data for missing features
train_weights = [w for item in train_items if (w := infer_weight(item)) is not None]
train_ranks   = [r for item in train_items if (r := infer_rank(item))   is not None]

average_weight = sum(train_weights) / len(train_weights) if train_weights else 1.0
average_rank   = sum(train_ranks)   / len(train_ranks)   if train_ranks   else 1_000_000.0

print(f"Average weight (lb): {average_weight:.3f}")
print(f"Average BSR rank   : {average_rank:,.0f}")

In [ ]:
def build_features(item):
    """Assemble a feature dictionary for a single item."""
    weight = infer_weight(item)
    rank   = infer_rank(item)
    return {
        "weight":      weight if weight is not None else average_weight,
        "rank":        rank   if rank   is not None else average_rank,
        "text_length": len(item.test_prompt()),
        "top_brand":   is_top_brand(item)
    }

In [ ]:
# Construct feature matrices for train and test splits
train_frame = pd.DataFrame([build_features(item) for item in train_items])
train_frame["price"] = [item.price for item in train_items]

test_frame = pd.DataFrame([build_features(item) for item in test_items[:250]])
test_frame["price"] = [item.price for item in test_items[:250]]

In [ ]:
# Baseline 3: Linear Regression on structured features
FEATURE_COLS = ["weight", "rank", "text_length", "top_brand"]
X_train = train_frame[FEATURE_COLS]
y_train = train_frame["price"]
X_test  = test_frame[FEATURE_COLS]
y_test  = test_frame["price"]

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)


def linear_baseline(item):
    return float(linear_model.predict(pd.DataFrame([build_features(item)]))[0])


Tester.test(linear_baseline, test_items[:250])

### Text-Based Baselines
Leverage the raw product description text via bag-of-words and word embeddings.

In [ ]:
# Baseline 4: Bag-of-Words + Linear Regression
document_texts = [item.test_prompt() for item in train_items]
price_targets  = np.array([item.price for item in train_items])

vectorizer = CountVectorizer(max_features=1000, stop_words="english")
X_matrix   = vectorizer.fit_transform(document_texts)

bow_model = LinearRegression()
bow_model.fit(X_matrix, price_targets)


def bow_predictor(item):
    pred = float(bow_model.predict(vectorizer.transform([item.test_prompt()]))[0])
    return max(pred, 0.0)  # Clamp to non-negative


Tester.test(bow_predictor, test_items[:250])

In [ ]:
# Baseline 5: Word2Vec embeddings + Linear SVR
processed_docs  = [simple_preprocess(text) for text in document_texts]
word2vec_model  = Word2Vec(
    sentences=processed_docs,
    vector_size=400,
    window=5,
    min_count=1,
    workers=4
)


def document_vector(text):
    """Average word vectors for all known tokens in the text."""
    words   = simple_preprocess(text)
    vectors = [word2vec_model.wv[w] for w in words if w in word2vec_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(word2vec_model.vector_size)


w2v_features = np.array([document_vector(text) for text in document_texts])
svr_model    = LinearSVR()
svr_model.fit(w2v_features, price_targets)


def w2v_predictor(item):
    return float(svr_model.predict([document_vector(item.test_prompt())])[0])


Tester.test(w2v_predictor, test_items[:250])

In [ ]:
# Baseline 6: Random Forest on structured features
forest_model = RandomForestRegressor(n_estimators=200, random_state=123, n_jobs=-1)
forest_model.fit(X_train, y_train)


def forest_predictor(item):
    return float(
        forest_model.predict(
            pd.DataFrame([build_features(item)])[FEATURE_COLS]
        )[0]
    )


Tester.test(forest_predictor, test_items[:250])

## GPT-4o-mini Fine-Tuning
Prepare training data in the OpenAI JSONL format and submit a fine-tuning job.

In [ ]:
# Subset used for fine-tuning (200 train, 50 validation)
fine_tune_train      = train_items[:200]
fine_tune_validation = train_items[200:250]

In [ ]:
def compose_messages(item, include_price=True):
    """
    Format an item as an OpenAI chat message list.

    When include_price=False, the assistant turn is truncated to the prefix
    only, suitable for inference-time prompting.
    """
    system_message    = "You estimate prices of items. Reply only with the price"
    user_prompt       = (
        item.test_prompt()
        .replace(" to the nearest dollar", "")
        .replace("\n\nPrice is $", "")
    )
    assistant_content = f"Price is ${item.price:.2f}" if include_price else "Price is $"

    return [
        {"role": "system",    "content": system_message},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": assistant_content}
    ]

In [ ]:
def build_jsonl(items):
    """Serialize a list of Items to an OpenAI fine-tuning JSONL string."""
    return "\n".join(json.dumps({"messages": compose_messages(item)}) for item in items)

In [ ]:
# Write JSONL files for the fine-tuning API
train_jsonl      = storage_dir / "balanced_pricer_train.jsonl"
validation_jsonl = storage_dir / "balanced_pricer_validation.jsonl"

train_jsonl.write_text(build_jsonl(fine_tune_train))
validation_jsonl.write_text(build_jsonl(fine_tune_validation))
print("JSONL files written.")

In [ ]:
# Upload files to OpenAI for fine-tuning
openai_client = OpenAI()

with open(train_jsonl, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")

with open(validation_jsonl, "rb") as f:
    validation_file = openai_client.files.create(file=f, purpose="fine-tune")

print(f"Train file ID     : {train_file.id}")
print(f"Validation file ID: {validation_file.id}")

In [ ]:
# Submit the fine-tuning job with W&B integration for experiment tracking
wandb_integration = {"type": "wandb", "wandb": {"project": "balanced-pricer"}}

fine_tune_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    seed=123,
    hyperparameters={"n_epochs": 1},
    integrations=[wandb_integration],
    suffix="balanced-pricer"
)

print(f"Job ID: {fine_tune_job.id}")
fine_tune_job

In [ ]:
# Poll job status and recent events -- re-run this cell until status is 'succeeded'
job_status = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id)
job_events = openai_client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=fine_tune_job.id, limit=10
)

print(f"Status: {job_status.status}")
for event in job_events.data:
    print(f"  [{event.created_at}] {event.message}")

In [ ]:
# Retrieve the fine-tuned model identifier once the job completes
fine_tuned_model_name = openai_client.fine_tuning.jobs.retrieve(fine_tune_job.id).fine_tuned_model
print(f"Fine-tuned model: {fine_tuned_model_name}")

In [ ]:
def tuned_predictor(item):
    """
    Query the fine-tuned GPT-4o-mini model for a price prediction.
    The model receives only the prompt prefix; the price token is extracted
    from the first assistant completion token(s).
    """
    messages = compose_messages(item, include_price=False)
    response = openai_client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=messages,
        seed=123,
        max_tokens=7
    )
    return get_price(response.choices[0].message.content)

In [ ]:
# Smoke test: single prediction vs. ground truth
if test_items:
    sample = test_items[0]
    print(f"Actual price   : ${sample.price:.2f}")
    print(f"Predicted price: ${tuned_predictor(sample):.2f}")

In [ ]:
# Full evaluation of the fine-tuned model against the test set
Tester.test(tuned_predictor, test_items[:250])